# 15 · Estimate each knockout's effect on each gene

A linear mixed model per response gene, with the knockouts as fixed effects and
the Leiden cluster as a random effect, so that a knockout's effect is estimated
across cell states rather than confounded with them.

It writes the full coefficient and p-value matrices, every knockout against
every response gene. Notebook 16 corrects them for multiple testing and
reduces them to what carries signal.

**Reads** `par_save_filename_8` and `par_em_selected_cells_file`, subsetting
the first to the cells listed in the second before fitting.
**Writes** `par_effect_coefs_file` and `par_effect_pvals_file`.

Fitted in blocks of `par_gene_block_size` genes, each written as it completes.

:::{note}
Two models are available, selected by `par_test_target_dist`.

`"NB"` fits a negative binomial mixed model, `lme4::glmer.nb` through `rpy2`,
on raw counts. Because `glmer.nb` cannot carry a thousand fixed effects, it
fits `par_target_block_size` knockouts at a time against a fixed panel of
`par_nb_target_control_cells` control cells plus the cells carrying that
block's knockouts.



## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
import statsmodels.formula.api as smf
import statsmodels.stats.multitest as smm
import warnings
warnings.filterwarnings("ignore")

EFFECT_DIR = "outputs/MixedEffectLM"
Path(EFFECT_DIR).mkdir(parents=True, exist_ok=True)
Path(par_effect_coefs_file).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# The per-gene object from notebook 13, subset to the cells the EM judged to be
# genuinely perturbed. A cell that carries a guide but shows no response to it
# would otherwise dilute every coefficient fitted below.
adata = sc.read(par_save_filename_8)
print(f"per-gene object: {adata.shape[0]} cells x {adata.shape[1]} genes")

selected = pd.read_csv(par_em_selected_cells_file).iloc[:, 0].astype(str)
adata = adata[adata.obs_names.isin(set(selected))].copy()
print(f"EM-selected     : {adata.shape[0]} cells ({len(selected)} in the list)")

if adata.shape[0] != len(set(selected)):
    print(f"WARNING: {len(set(selected)) - adata.shape[0]} listed cells are not in the object")

# GENE_CONTROL_ is the reference level. Every cell carries exactly one
# indicator, so keeping it alongside the intercept would make the design
# singular.
knockouts = [c for c in adata.uns["feature_barcode_names_filtered_GENES"]
             if c != "GENE_CONTROL_"]

# The design is built as an array rather than a formula. With a thousand
# covariates a formula string cannot be parsed, and building the matrix
# directly also sidesteps the gene names that contain a hyphen.
terms = knockouts + ["n_genes", "mt_frac"]
exog = adata.obs[terms].astype(float).to_numpy()
exog = np.column_stack([np.ones(exog.shape[0]), exog])
exog_names = ["Intercept"] + terms

groups = adata.obs["leiden"].to_numpy()
gene_names = list(adata.var_names)

print(f"knockouts: {len(knockouts)}, response genes: {len(gene_names)}")
print(f"design: {exog.shape[0]} cells x {exog.shape[1]} columns "
      f"(intercept + {len(knockouts)} knockouts + 2 quality covariates)")
print(f"random intercept: leiden, {len(set(groups))} groups")
print(f"model: {par_test_target_model} ({par_test_target_dist})")

## Fit one mixed model per gene

In [ ]:
if par_test_target_dist == "NB":
    # Negative binomial path. glmer.nb cannot carry a thousand fixed effects, so
    # knockouts are fitted par_target_block_size at a time against a fixed panel of
    # control cells, and on raw counts rather than normalised expression.
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri, numpy2ri
    from rpy2.robjects.conversion import localconverter

    Path(par_effect_nb_dir).mkdir(parents=True, exist_ok=True)

    ro.r('''
    library("MASS")
    library("lme4")

    fit_nb_block <- function(design_df, expr_mat, model_formula, resp_names) {
        out <- data.frame()
        for (j in seq_len(ncol(expr_mat))) {
            tryCatch({
                design_df[["y"]] <- expr_mat[, j]
                fit <- glmer.nb(as.formula(model_formula), data = design_df,
                                control = glmerControl(optimizer = "bobyqa",
                                                       calc.derivs = FALSE))
                tab <- data.frame(coefficients(summary(fit)))
                tab$coefName <- rownames(tab)
                tab$respGene <- resp_names[j]
                out <- rbind(out, tab)
            }, error = function(e) message(paste("  skipped", resp_names[j], ":",
                                                 conditionMessage(e))))
        }
        out
    }
    ''')

    raw = adata.raw.X if adata.raw is not None else adata.X
    raw = raw.toarray() if hasattr(raw, "toarray") else raw
    if adata.raw is not None:
        raw = pd.DataFrame(raw, index=adata.obs_names,
                           columns=adata.raw.var_names)[adata.var_names].to_numpy()

    is_control = (adata.obs["GENE_CONTROL_"] == 1).to_numpy()
    control_rows = np.where(is_control)[0][:par_nb_target_control_cells]
    print(f"control panel: {len(control_rows)} cells")

    for block_start in range(0, len(knockouts), par_target_block_size):
        block_stop = min(block_start + par_target_block_size, len(knockouts))
        block = knockouts[block_start:block_stop]
        out = f"{par_effect_nb_dir}/coefs_{block_start}_{block_stop}_MixedEffectNB.csv"
        if Path(out).exists():
            continue

        carries = adata.obs[block].sum(axis=1).to_numpy() > 0
        rows = np.union1d(control_rows, np.where(carries)[0])

        design = adata.obs.iloc[rows][block + ["n_genes", "mt_frac", "leiden"]].copy()
        design.columns = [c.replace("-", "") for c in design.columns]
        terms_b = [c.replace("-", "") for c in block]
        formula = "y~" + "+".join(terms_b + ["n_genes", "mt_frac"]) + " + (1 | leiden)"

        with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
            fitted = ro.globalenv["fit_nb_block"](
                design, raw[rows, :], formula, ro.StrVector(gene_names))
            fitted = ro.conversion.rpy2py(fitted)

        fitted.to_csv(out, index=False)
        print(f"  knockouts {block_start}-{block_stop}: {len(rows)} cells, {len(fitted)} rows")

    print("all knockout blocks fitted")

else:
    import statsmodels.api as sm

    # Expression itself is the response: the quality covariates are fixed effects
    # and the cluster is a random intercept, so fitting the cluster-residual layer
    # would remove the cluster twice.
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

    for start in range(0, len(gene_names), par_gene_block_size):
        stop = min(start + par_gene_block_size, len(gene_names))
        out = f"{EFFECT_DIR}/coefs_{start}_{stop}_MixedEffect.csv"
        if Path(out).exists():
            continue

        coefs, pvals = pd.DataFrame(), pd.DataFrame()
        for j in range(start, stop):
            fit = sm.MixedLM(endog=X[:, j], exog=exog, groups=groups).fit(method=["bfgs"])
            coefs[gene_names[j]] = pd.Series(fit.params[:len(exog_names)].to_numpy()
                                             if hasattr(fit.params, "to_numpy") else fit.params[:len(exog_names)],
                                             index=exog_names)
            pvals[gene_names[j]] = pd.Series(fit.pvalues[:len(exog_names)].to_numpy()
                                             if hasattr(fit.pvalues, "to_numpy") else fit.pvalues[:len(exog_names)],
                                             index=exog_names)

        coefs.to_csv(out)
        pvals.to_csv(f"{EFFECT_DIR}/pvalues_{start}_{stop}_MixedEffect.csv")
        print(f"  genes {start}-{stop} done")

    print("all blocks fitted")


## Assemble and FDR correct

In [ ]:
def load_blocks(prefix):
    frames = []
    for start in range(0, len(gene_names), par_gene_block_size):
        stop = min(start + par_gene_block_size, len(gene_names))
        frames.append(pd.read_csv(f"{EFFECT_DIR}/{prefix}_{start}_{stop}_MixedEffect.csv",
                                  index_col=0))
    return pd.concat(frames, axis=1)

# every fitted row is kept, including the intercept, the quality covariates and
# the random-effect variance. Notebook 16 drops them when it corrects for
# multiple testing.
coefs = load_blocks("coefs")
pvals = load_blocks("pvalues")

coefs.to_csv(par_effect_coefs_file)
pvals.to_csv(par_effect_pvals_file)

print(f"coefficients: {coefs.shape[0]} rows x {coefs.shape[1]} genes "
      f"({len(knockouts)} knockouts plus the intercept, quality covariates "
      f"and group variance)")
print(f"written: {par_effect_coefs_file}")
print(f"written: {par_effect_pvals_file}")
print()
print("Notebook 16 corrects these for multiple testing and reduces them to the")
print("knockouts and genes that carry signal.")